# Self-Diffusion Coefficient Extraction (Adaptive Linear Fit)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re, sys
from tqdm.auto import tqdm

import MDAnalysis as mda
from MDAnalysis.analysis import msd

In [ ]:
BASE = Path("../data/MD/selfdiff/gaff2/simulation")
TIMESTEP_PS = 0.5          # frame spacing in ps; edit if different
from datetime import datetime
PLOT_DIR = Path(f"msd_plots_adaptive_full_{datetime.now():%Y%m%d-%H%M%S}")
PLOT_DIR.mkdir(exist_ok=True)

In [ ]:
# adaptive-window parameters
MIN_LEN  = 300       # frames  (lower bound)
MAX_LEN  = 4000      # frames  (upper bound)
STEP     = 50        # frames  (grid spacing)
SKIP_TIME_PS  = 50.0 # skip first 50 ps as ballistic regime
R2_THRESH = 0.995    # linearity threshold

In [ ]:
from numpy.linalg import lstsq

def diffusion_adaptive(tag: str):
    """
    Returns D (m²/s) and window length (ps) using:
      • skip first SKIP_TIME_PS
      • scan window lengths MIN_LEN..MAX_LEN (STEP)
      • choose longest window with R² ≥ R2_THRESH
    """
    prmtop = BASE / f"{tag}.prmtop"
    traj   = BASE / f"{tag}_prod_unwrapped.nc"

    # --- load trajectory & MSD ---
    u = mda.Universe(prmtop, traj)
    MSD = msd.EinsteinMSD(u, msd_type="xyz", fft=True)
    MSD.run()
    msd_vals = MSD.results.timeseries
    lags     = np.arange(MSD.n_frames) * TIMESTEP_PS

    # --- absolute-time skip ---
    start = int(np.ceil(SKIP_TIME_PS / TIMESTEP_PS))
    msd_vals, lags = msd_vals[start:], lags[start:]
    nframes = len(lags)
    if nframes < MIN_LEN:
        raise ValueError(f"{tag}: too short after {SKIP_TIME_PS} ps skip")

    # --- search longest linear window ---
    best = {"r2": -np.inf}
    for w in range(min(MAX_LEN, nframes), MIN_LEN - 1, -STEP):   # longest→shortest
        for i in range(0, nframes - w + 1):
            x = lags[i:i+w]; y = msd_vals[i:i+w]
            A = np.vstack([x, np.ones_like(x)]).T
            slope, intercept = lstsq(A, y, rcond=None)[0]
            r2 = 1 - ((y - (intercept + slope*x))**2).sum() / ((y - y.mean())**2).sum()
            if r2 >= R2_THRESH:
                best = {"i0": i, "w": w, "slope": slope, "r2": r2}
                break
        if best["r2"] >= R2_THRESH:       # longest satisfactory window found
            break

    if best["r2"] < R2_THRESH:
        raise RuntimeError(f"{tag}: no segment reached R² ≥ {R2_THRESH}")

    # --- diffusion coefficient ---
    D_m2_s   = (best["slope"] / 6.0) * (1e-10)**2 / 1e-12
    window_ps = (best["w"] - 1) * TIMESTEP_PS

    # --- plot & save ---
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(lags, msd_vals, lw=0.8, label="MSD (all)")
    ax.plot(lags[best["i0"]:best["i0"]+best["w"]],
            msd_vals[best["i0"]:best["i0"]+best["w"]],
            lw=2.2, label=f"fit window (R²={best['r2']:.3f})")
    ax.set_xlabel("Time (ps)"); ax.set_ylabel("MSD (Å²)")
    ax.set_title(f"MSD – {tag}")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"MSD_{tag}.png", dpi=300)
    plt.close(fig)

    return D_m2_s, window_ps

In [ ]:
tags = sorted({
    re.sub(r'_prod_unwrapped\.nc$', '', f.name)
    for f in BASE.glob('*_prod_unwrapped.nc')
})
print(f"Found {len(tags)} trajectories")

In [ ]:
tags_new = [t for t in tags if int(t.split('_')[0]) > 494 and t.endswith('_35')]
print(f"Found {len(tags_new)} more trajectories")

In [ ]:
df_md = pd.read_csv("../data/MD/selfdiff/gaff2/diffusion_coefficients_adaptive_full.csv")
len(df_md)

In [ ]:
records = []
for tag in tqdm(tags_new, desc="Processing"):
    try:
        D, win_ps = diffusion_adaptive(tag)
        mol, box = tag.rsplit('_', 1)
        records.append({"molecule": mol,
                        "box": int(box),
                        "D_m2_s": D,
                        "window_ps": win_ps})
    except Exception as e:
        print(f"{e}", file=sys.stderr)

In [ ]:
df = pd.DataFrame(records).sort_values(["molecule", "box"])
df

In [ ]:
df.window_ps.describe()

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
outpath = f"../data/MD/selfdiff/gaff2/diffusion_coefficients_adaptive_full.csv"
df.to_csv(outpath, index=False)
print(f"{len(df)} successes")